# TypeSafe AI 入门实验（AI Primer Lab）

针对官方文档对应章节的可运行实验笔记，全部实验使用**中文场景与中文提示词**。
面向会基础 Python、刚接触 AI Agent 的读者。

**学习目标：** 理解文档中的三种训练目标，用手工数据解释校准，并把数学演示与真实模型观测分开。

[官方原文](https://docs.typesafe.ai/introduction/machine-learning-primer) · [中文参考](https://bald0wang.github.io/jev-docs-zh/introduction/machine-learning-primer/)。本章以中文重述理论、复刻对应场景；扩展实验会单独说明。
所有客户、订单及消息均为教学合成数据。

## 笔记本结构

| 章节 | 内容 |
|---|---|
| 0. 准备 | 安装库、配置客户端、连通性测试与离线示例 |
| 1 | RLHF、RLVR、RLCD 与机器接口 |
| 2 | 人工概率实验 |
| 3 | 三条中文消息的 Noul 探针 |
| 练习与小结 | 练习、自查、总结与本次执行记录 |

实验按**原理 → 理论根基 → 定义数据 → 定义问题 → 调用 → 解读结果**展开，每个代码单元格只做一件事。

## 运行要求

- Python ≥ 3.10；本章使用 `typesafe-sdk==0.7.0`。
- 真实实验需要启动进程的 `TYPESAFE_API_KEY` 环境变量，密钥不要写进 Notebook。

在本仓库 `notebooks/` 目录创建环境并打开本文件：

```bash
./setup_env.sh
.venv/bin/python -m pip install -r requirements.txt -c generators/constraints-foundations.txt
.venv/bin/jupyter lab ai_primer_experiments.ipynb
```

产品名、字段名和选项 key 保持英文，state、提示词与解说使用中文。
默认 `JEV_RUN_MODE=live`，调用失败即停止；无密钥学习时，在启动 Jupyter 前设置 `JEV_RUN_MODE=offline`。
`auto` 仅供教学体验，缺密钥或 401 时显式回退；正式验收使用 `live`。

**验证状态：真实 API 待验收。** 本文件尚未执行真实 API；离线检查仅验证代码路径。
批量执行、离线预览和验收记录见本目录 `MAINTENANCE.md`。

## 0. 准备

本节可折叠阅读，但独立运行时不能跳过。客户端、辅助对象和示例数据都在本文件中定义。

### 0.1 安装依赖

推荐先运行 `setup_env.sh`。只有当前内核缺少 SDK 时，本格才安装依赖。

In [ ]:
import importlib.util
if importlib.util.find_spec("typesafe_sdk") is None:
    %pip install -q typesafe-sdk==0.7.0

**观察与理解：** 安装包的名字是 typesafe-sdk，Python 导入名是 typesafe_sdk。安装成功不代表 API 已连通。

### 0.2 导入与配置

默认模型固定版本，便于记录实验条件；可通过环境变量更换。不要从 Notebook 输入密钥。

In [ ]:
import os
import json
import time
import math
from datetime import datetime, timezone
from importlib.metadata import version
from typesafe_sdk import (
    Choice, Score, Noul, NoulCriteria, TypeSafeClient,
    TypeSafeAuthenticationError, RetryPolicy,
)

MODEL = os.environ.get("TYPESAFE_DEFAULT_MODEL", "jev-1.13.0")
RUN_MODE = os.environ.get("JEV_RUN_MODE", "live")
API_KEY = os.environ.get("TYPESAFE_API_KEY", "")
if RUN_MODE not in {"live", "offline", "auto"}:
    raise ValueError("JEV_RUN_MODE 只能是 live、offline 或 auto")
if RUN_MODE == "live" and not API_KEY:
    raise RuntimeError("请在启动 Jupyter 前配置 TYPESAFE_API_KEY 环境变量")
client = None
if RUN_MODE != "offline" and API_KEY:
    client = TypeSafeClient(api_key=API_KEY, model=MODEL, timeout=30,
                           retry=RetryPolicy(max_retries=0))
print("模式：", RUN_MODE, "SDK：", version("typesafe-sdk"), "模型配置：", MODEL)

正式验收禁用自动回退，且不自动重试，以便请求数量有界。`auto` 与 `offline` 是教学工具，不代表成功连接模型。

### 0.3 连通性测试

用一条 Noul 检查真实响应能否返回。网络、限流与输入错误直接抛出，不伪装成不确定判断。

In [ ]:
PING = {"source": "offline", "reason": "未发起连通性请求"}
if client is not None:
    try:
        ping = client.system_one("你好", {"greeting": Noul(
            instructions="这段文字是否在打招呼？")})
        PING = {"source": "live", "model": ping.model,
                "input_tokens": ping.usage.input_tokens,
                "output_tokens": ping.usage.output_tokens}
    except TypeSafeAuthenticationError:
        if RUN_MODE == "live":
            raise
        client.close()
        client = None
        PING["reason"] = "401 鉴权失败，仅教学模式允许回退"
print(json.dumps(PING, ensure_ascii=False))

**观察与理解：** source=live 表示这一次连通性请求成功；仍要查看后续实验记录，不能用它代替整章验收。

### 0.4 离线替身

沿用参考模板的 `_FakeAnswer` 与 `_FakeResponse` 访问方式。人工数字仅用来检验读取字段和代码分支。

In [ ]:
class _FakeAnswer:
    def __init__(self, type_, **values):
        self.type = type_
        for name, value in values.items():
            setattr(self, name, value)


class _FakeResponse:
    def __init__(self, answers):
        self.answers = answers
        self.choices = {k: v for k, v in answers.items() if v.type == "choice"}
        self.scores = {k: v for k, v in answers.items() if v.type == "score"}
        self.nouls = {k: v for k, v in answers.items() if v.type == "noul"}
        self.model = "人工示例，非模型预测"
        self.usage = _FakeAnswer("usage", input_tokens=0, output_tokens=0)

人工 Score 由概率计算期望，避免模板中的分数与分布不一致。人工 confidence 只是指定的演示字段，不是在复现服务端的计算公式。

定义两种示例答案构造器；Noul 可直接用 `_FakeAnswer`。所有具体答案集中在下一节。

In [ ]:
def fake_choice(probabilities, confidence):
    return _FakeAnswer("choice", choice=max(probabilities, key=probabilities.get),
                       probabilities=probabilities, confidence=confidence)


def fake_score(probabilities, legend, confidence):
    return _FakeAnswer("score", score=sum(k * p for k, p in probabilities.items()),
                       probabilities=probabilities, legend=dict(enumerate(legend)),
                       confidence=confidence)

**观察与理解：** 例如概率 {0:0.05, 1:0.26, 2:0.69} 对应 1.64。不能把另一个数与该分布配在一起。

### 0.5 统一调用入口

每次调用记录来源、模型与 token 用量。离线耗时记为 None，不把本地字典访问当成模型速度。

In [ ]:
CALL_LOG = []


class TS:
    def call(self, state, questions, offline_answers, label):
        start = time.perf_counter()
        source = "live"
        if client is None:
            response, source = _FakeResponse(offline_answers), "offline"
        else:
            try:
                response = client.system_one(state, questions)
            except TypeSafeAuthenticationError:
                if RUN_MODE != "auto":
                    raise
                response, source = _FakeResponse(offline_answers), "offline"
        validate_response(response, questions)
        CALL_LOG.append({"case": label, "source": source, "model": response.model,
                         "seconds": time.perf_counter() - start if source == "live" else None,
                         "input_tokens": response.usage.input_tokens,
                         "output_tokens": response.usage.output_tokens})
        if source == "offline":
            print("离线示例：", label, "；人工答案，不是 Jev 实测")
        return response


ts = TS()

与参考模板相比，这里增加了严格 live 模式和逐次记录。保留 401 教学回退，但超时、429 等错误继续失败，防止验收被回退掩盖。

校验结构与数值契约；只断言接口应满足的性质，不断言真实模型必须预测某个标签。

In [ ]:
def validate_response(response, questions):
    if set(response.answers) != set(questions):
        raise ValueError("答案 ID 与问题 ID 不一致")
    for key, question in questions.items():
        answer = response.answers[key]
        if isinstance(question, Noul):
            if not 0 <= answer.noul <= 1:
                raise ValueError("Noul 超出概率范围")
            continue
        probabilities = answer.probabilities
        if not all(math.isfinite(p) and 0 <= p <= 1 for p in probabilities.values()):
            raise ValueError("概率值无效")
        if not math.isclose(sum(probabilities.values()), 1, abs_tol=0.02):
            raise ValueError("概率之和偏离 1")
        if not 0 <= answer.confidence <= 1:
            raise ValueError("confidence 超出范围")
        if isinstance(question, Choice):
            if set(probabilities) != set(question.criteria):
                raise ValueError("Choice 选项集合不一致")
            if answer.choice not in probabilities:
                raise ValueError("Choice 标签不在选项中")
        else:
            expected = sum(int(k) * p for k, p in probabilities.items())
            if not math.isclose(answer.score, expected, abs_tol=0.03):
                raise ValueError("Score 与概率加权期望不一致")

**观察与理解：** 容差用于服务端数值舍入。结构检查通过只说明响应可读取，不证明语义判断正确。

显示结果时统一列出类型、概率和置信度；Noul 不额外制造 confidence 字段。

In [ ]:
def show(response):
    rows = {}
    for key, answer in response.answers.items():
        rows[key] = {name: getattr(answer, name) for name in
                     ("type", "choice", "score", "noul", "confidence", "probabilities", "legend")
                     if hasattr(answer, name)}
    print(json.dumps(rows, ensure_ascii=False, indent=2))

### 0.6 本章离线示例数据

以下数值全部人工构造，专门测试分支；不来自 Jev，也不能用于估计中文准确率或校准情况。正式 live 运行不会使用这些答案。

In [ ]:
REFUND_OFFLINE = [
    {"refund_requested": _FakeAnswer("noul", noul=0.97)},
    {"refund_requested": _FakeAnswer("noul", noul=0.03)},
    {"refund_requested": _FakeAnswer("noul", noul=0.43)},
]

## 1. 📖 理论根基：训练目标与输出契约

机器原生智能是 TypeSafe 对产品方向的描述：软件需要容易解析、检查、测试和记录的输出。
文档关于未来机器交互占比的说法属于愿景，本教程不把它当作已验证的行业统计。

| 方法 | 名称 | 文档强调的优化目标 |
|---|---|---|
| RLHF | 基于人类反馈的强化学习 | 人们偏好的回答 |
| RLVR | 使用可验证奖励的强化学习 | 能由规则或评估器验证的结果 |
| RLCD | 面向校准决策的强化学习 | TypeSafe 描述的决策与概率输出 |

这张表用于理解产品文档，不是断言所有聊天模型都只有一种训练方法。
人类偏好与业务可靠性并不等价；动听、自信的说法仍可能缺少依据。
文档借 mode dropping 说明偏好优化可能缩窄输出多样性，不能由此推断每个 RLHF 模型必然不可靠。

## 2. 数学演示：0.8 到底意味着什么

### 📖 理论根基

考虑一批事件，其中每件事都被预测有 0.8 的概率发生。若长期发生频率也接近 80%，这是校准的证据之一。
单条预测发生或没发生，都不能独立证明整套系统校准或失准。有限样本还会有抽样波动。

下面是**人工构造的数学数据**，不调用 API，不代表 Jev 的测试集或模型效果。


[官方原文](https://docs.typesafe.ai/confidence) · [中文参考](https://bald0wang.github.io/jev-docs-zh/confidence/)

先构造两个概率组，各 20 条事件。1 表示事件发生，0 表示没有发生。

In [ ]:
MATH_GROUPS = {
    0.2: [1] * 4 + [0] * 16,
    0.8: [1] * 16 + [0] * 4,
}

计算每组的平均预测概率与发生频率。

In [ ]:
calibration_rows = [
    {"预测概率": probability, "样本数": len(labels),
     "发生频率": sum(labels) / len(labels)}
    for probability, labels in MATH_GROUPS.items()
]

**观察与理解：** 这两个组被特意构造成频率匹配，目的是解释定义。不能把该结果归功于模型。

显示结果并观察组内仍然存在的失败。

In [ ]:
print(json.dumps(calibration_rows, ensure_ascii=False, indent=2))
print("预测为 0.8 的这一组，仍有", MATH_GROUPS[0.8].count(0), "次事件没有发生")

**观察与理解：** 校准良好与单次必然正确是两回事。预测所有事件都为整体基率也可能校准，却不擅长区分个例。

扩展：用 Brier 分数同时观察概率与二元标签的距离。越小越好，但它不是纯校准误差。

In [ ]:
def brier(probabilities, labels):
    if not labels or len(probabilities) != len(labels):
        raise ValueError("概率与标签必须非空且等长")
    return sum((p - y) ** 2 for p, y in zip(probabilities, labels)) / len(labels)


LABELS = MATH_GROUPS[0.2] + MATH_GROUPS[0.8]
PROBABILITIES = [0.2] * 20 + [0.8] * 20
OVERCONFIDENT = [0.01] * 20 + [0.99] * 20

比较两个手工预测器。

In [ ]:
print({"匹配组内频率": brier(PROBABILITIES, LABELS),
       "人为推向极端": brier(OVERCONFIDENT, LABELS)})

**观察与理解：** 同样的事件标签下，把概率推向极端可能更差。真实评测还要报告数据来源、样本量、分组与任务范围。

## 3. 小型探针：明确、否定与模糊的退款请求

原理：观察同一个命题对不同消息返回什么概率。这里新增三条教学场景，帮助理解 AI 入门中的概率概念。
三条消息不足以评估校准；它们只是接口与行为观察。模糊消息也不保证落入某个置信区间。

准备三条中文消息。

In [ ]:
MESSAGES = [
    "订单扣了两次款，请退还重复扣取的那一笔。",
    "我不需要退款，只想问下一张发票什么时候出。",
    "这个扣款你们看着处理一下吧。",
]

定义任务：只判断是否明确提出退款，不猜测未表达的意图。

In [ ]:
REFUND_QUESTIONS = {
    "refund_requested": Noul(instructions="客户是否明确要求退款或退还款项？仅抱怨扣款不算明确请求。"),
}

**观察与理解：** 三个输入是三个 state，需要三次调用。它们不是对同一个 state 提三个不同问题。

依次执行探针。

In [ ]:
refund_responses = [
    ts.call(message, REFUND_QUESTIONS, REFUND_OFFLINE[i], f"退款措辞探针 {i + 1}")
    for i, message in enumerate(MESSAGES)
]

记录真实返回，不预填‘模型一定降低置信度’的结论。

In [ ]:
for message, response in zip(MESSAGES, refund_responses):
    print({"消息": message, "退款请求概率": response.nouls["refund_requested"].noul})

**观察与理解：** 观察明确否定与明确请求是否区分开；模糊消息的数值应原样记录。Noul 的不确定性直接看概率，不能访问不存在的 confidence。

## 练习与自查

为什么不能把本章三条消息重复请求一百次，然后声称完成了中文退款意图的校准评估？

<details><summary>参考思路：先完成练习再展开</summary>

重复请求只覆盖三个输入，不能代表业务输入分布。应收集独立样本、定义标注标准、保留未用于调题的测试集，并按概率分组报告样本量和频率。

</details>

## 小结

| 材料 | 可以支持什么结论 |
|---|---|
| 人工概率表 | 校准定义与数学计算 |
| 三条 live 探针 | 本次输入下的实际模型行为 |
| 独立标注的大样本 | 才适合进一步评估校准与业务效果 |

下一章：[System One](system_one_experiments.ipynb)。数学实验属于教学扩展，不是在训练或复现 RLCD。

离线运行只说明教材代码能执行。正式交付必须实际运行 live，并阅读每条输出；缺失的分支应记为未观察到。

## 本次执行记录

先关闭连接，再生成记录。下面的 JSON 由实际运行计算，批量执行器会据此检查来源。

In [ ]:
if client is not None:
    client.close()

真实探针只演示行为路径；若据其返回挑选样例，这批样例就不适合再当作无偏准确率测试集。延迟也只是本次网络环境中的观测。

In [ ]:
AUDIT = {
    "kind": "jev_execution_audit",
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "sdk": version("typesafe-sdk"), "requested_model": MODEL,
    "mode": RUN_MODE, "ping": PING,
    "real_calls": sum(x["source"] == "live" for x in CALL_LOG),
    "offline_calls": sum(x["source"] == "offline" for x in CALL_LOG),
    "cases": CALL_LOG,
    "coverage": globals().get("COVERAGE", {}),
    "validation_status": "live_executed_requires_review" if (
        PING["source"] == "live" and CALL_LOG
        and all(x["source"] == "live" for x in CALL_LOG)
    ) else "offline_only_not_model_evidence",
}
print(json.dumps(AUDIT, ensure_ascii=False, indent=2))

读完输出后，在本仓库 `notebooks/MAINTENANCE.md` 的验收表中记录日期、真实模型、观察到的分支和偏离预期之处。不要把人工演示数值抄进实测记录。